# Lesson 3 — Attention (runnable)

Every word looks at every other word. Three steps:
1. **Scores** — dot product every pair of words
2. **Softmax** — turn scores into a 100% attention budget per word
3. **Mix** — each word's new vector = weighted blend of all word vectors

Runnable version of [`03_attention.py`](../03_attention.py). For the worked 2-word "the cat" example by hand, see [`03_walkthrough.md`](../03_walkthrough.md).


## Imports

In [ ]:
import torch                                # PyTorch tensors.
import torch.nn.functional as F             # F.softmax used below.

torch.manual_seed(0)
torch.set_printoptions(precision=2, sci_mode=False)

## Set up some fake "embeddings" for a 4-word sentence

In [ ]:
words = ["the", "dog", "chased", "cat"]
d = 4                                       # Vector dimension.
x = torch.randn(len(words), d)              # Random "embeddings". Shape (4, 4).

print("Input vectors x (one row per word):")
print(x)

## Step 1 — Scores (every pair of words)

`x @ x.T` computes all 16 pair-wise dot products at once. Big number = related.

In [ ]:
scores = x @ x.T                            # Shape (4, 4).
print("Scores (raw dot products):")
print(scores)

## Step 2 — Softmax → 100% attention budget per word

In [ ]:
attention = F.softmax(scores, dim=-1)       # Each row sums to 1.

print("Attention weights (each row is a probability distribution):")
print("              " + "  ".join(f"{w:>7s}" for w in words))
for i, w in enumerate(words):
    print(f"  {w:10s}  " + "  ".join(f"{a:7.3f}" for a in attention[i]))

## Step 3 — Mix (weighted blend of word vectors)

In [ ]:
new_x = attention @ x                       # Weighted sum.
print("Output vectors (after attention):")
print(new_x)

## Real attention: Q, K, V projections

The naïve version uses the same vector for "what am I looking for" and "what do I offer". Real attention separates these with three learned matrices:

- **Q (query)** — what am I looking for?
- **K (key)** — what do I offer to others?
- **V (value)** — what do I contribute if matched?

Each is `x` multiplied by a learned matrix. Plus we divide scores by `sqrt(d)` to keep them in a sensible range.

In [ ]:
Wq = torch.randn(d, d)                      # Query projection.
Wk = torch.randn(d, d)                      # Key projection.
Wv = torch.randn(d, d)                      # Value projection.

Q = x @ Wq
K = x @ Wk
V = x @ Wv

scores = Q @ K.T / (d ** 0.5)               # Scaled dot product.
attention = F.softmax(scores, dim=-1)
output = attention @ V

print("Scaled dot-product attention output:")
print(output.detach())

**That's it.** The formula at the heart of GPT, BERT, PRAGMA:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d}}\right) V$$

## Things to try

1. Add two more words for a 6-word sentence. What's the attention matrix shape?
2. Set `x[1] = x[3]` (make two words identical). What do you notice in the attention matrix?
3. Print `scores` and `attention` side by side. Notice how softmax amplifies the biggest scores.
4. Drop the `/ sqrt(d)` scaling. What happens to attention sharpness?